# 06 — Frontier-Model Validation: Variants A and E on GPT-5.5

**Purpose.** Test whether the *Instruction Overload* effect observed for GPT-4o-mini in `02_run_ablation_hinted.ipynb` persists on a current frontier model. We re-run the two endpoints of the complexity spectrum — Variant A (Baseline, the simplest prompt) and Variant E (Full Framework, the most composite prompt) — on GPT-5.5, using the same HINTED prompts and the same stratified sample used elsewhere in the study. If the Full Framework regresses below the Baseline on GPT-5.5 as it did on GPT-4o-mini, the Instruction Overload effect is shown to be cross-generational rather than a small-model artefact.

**Design rationale.** This is a *cross-generation* validation (GPT-4o-mini is a GPT-4-generation model; GPT-5.5 is a 2026 frontier reasoning model), not a same-tier comparison. We deliberately restrict the variant set to A and E, the two extremes of prompt complexity, because the Instruction Overload claim is specifically about the A-versus-E contrast. Variants B, C, and D are not re-run here.

**Sample.** By default we use the **same 100-sample stratified subset** committed as `data/file_list_100_stratified.txt` (50 CWE-89 SQLi, 42 CWE-78 CmdInj, 8 Safe), so that the frontier-model numbers are directly comparable to the RQ3 cross-generation results. A `SAMPLE_SIZE_OVERRIDE` parameter is provided if a smaller 50-sample run is preferred for cost reasons; if set, the subset is drawn deterministically (fixed seed) while preserving category proportions.

**Inputs.**
- `<BASE_DIR>/final_dataset/` — produced by `01_dataset_curation.ipynb`.
- `data/file_list_100_stratified.txt` — committed to the repository.
- `prompts/variant_A_baseline.txt`, `prompts/variant_E_full_hinted.txt` — committed to the repository (identical to those used in `02_run_ablation_hinted.ipynb`).
- `prompts/system_message.txt` — committed to the repository.
- An OpenAI API key with access to `gpt-5.5`.

**Outputs.**
- `<BASE_DIR>/results/rq4_frontier_gpt55.csv` — one row per sample, with columns `Variant_A_Baseline` and `Variant_E_Full` containing the model's prediction (`Vulnerable` / `Safe`).

**Cost & runtime.** With the default 100-sample subset: 100 samples × 2 variants = 200 API calls on `gpt-5.5`. Approximate cost: USD 1.60–2.80 (GPT-5.5 is priced at $5.00 / 1M input and $30.00 / 1M output as of April 2026). Approximate runtime: 15–30 minutes depending on reasoning-token volume and network latency. With `SAMPLE_SIZE_OVERRIDE = 50`: 100 API calls, approx. USD 0.80–1.40. The script supports resume-on-interrupt.

**Note on the JSON response format.** GPT-5.5 is a reasoning model. We keep `response_format={'type': 'json_object'}` to match the protocol used for GPT-4o-mini, and we retain the same system message and prompts so that the only changed variable is the model. If GPT-5.5 rejects or ignores the `temperature` parameter (some reasoning models do), the setup cell falls back to omitting it; see the note in the run cell.


## 0. Confirm the correct model string (run this first)

OpenAI model identifiers change over time and may include a date-stamped snapshot (e.g., `gpt-5.5` vs `gpt-5.5-2026-04-23`). This cell lists the models your API key can access and filters for GPT-5.5 variants, so you can copy the exact string into `MODEL_ID` in the next cell. If the list is long, look for the shortest `gpt-5.5` entry without a `-preview` or `-chat` suffix for the standard model.

In [ ]:
import os
from openai import OpenAI

api_key = os.environ.get('OPENAI_API_KEY')
if not api_key:
    from getpass import getpass
    api_key = getpass('Enter your OpenAI API key (will not be stored): ')
client = OpenAI(api_key=api_key)

# List available models and filter for GPT-5.5 candidates.
available = sorted(m.id for m in client.models.list())
gpt55 = [m for m in available if m.startswith('gpt-5.5')]

print('GPT-5.5 model strings available to your key:')
if gpt55:
    for m in gpt55:
        print(f'  {m}')
else:
    print('  (none found — check that your key/project has GPT-5.5 access)')
    print('\nAll GPT-5.x models available:')
    for m in available:
        if m.startswith('gpt-5'):
            print(f'  {m}')


## 1. Setup

Set `MODEL_ID` to the exact string confirmed in the previous cell. Edit `BASE_DIR_OVERRIDE` if you want to override the auto-located workspace directory. Set `SAMPLE_SIZE_OVERRIDE = 50` if you want the cheaper 50-sample run instead of the default 100; leave it as `None` to use the full 100-sample stratified subset.

In [ ]:
import os
import json
import time
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI

# ---- USER-EDITABLE ----
BASE_DIR_OVERRIDE    = None       # e.g., Path('/content/drive/MyDrive/llm-vuln-detection-ablation')
MODEL_ID             = 'gpt-5.5'  # <-- paste the exact string confirmed in cell 0 if different
SAMPLE_SIZE_OVERRIDE = None       # None = use all 100 stratified samples; or set to 50
TEMPERATURE          = 0.1        # ignored automatically if the model rejects it (see run cell)
STRATIFY_SEED        = 42         # used only when SAMPLE_SIZE_OVERRIDE subsamples
MAX_RETRIES          = 3
RETRY_BACKOFF_SEC    = 2
# -----------------------

def find_repo_root() -> Path:
    """Locate the repository root by walking upward from the current working directory."""
    sentinels = ('README.md', 'requirements.txt', '.git')
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if any((candidate / s).exists() for s in sentinels):
            return candidate
    return cwd

REPO_ROOT       = find_repo_root()
BASE_DIR        = Path(BASE_DIR_OVERRIDE).resolve() if BASE_DIR_OVERRIDE else (REPO_ROOT / 'workspace')
DATASET_DIR     = BASE_DIR / 'final_dataset'
RESULTS_DIR     = BASE_DIR / 'results'
PROMPTS_DIR     = REPO_ROOT / 'prompts'
DATA_DIR        = REPO_ROOT / 'data'
OUTPUT_CSV      = RESULTS_DIR / 'rq4_frontier_gpt55.csv'
STRATIFIED_LIST = DATA_DIR / 'file_list_100_stratified.txt'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

api_key = os.environ.get('OPENAI_API_KEY')
if not api_key:
    from getpass import getpass
    api_key = getpass('Enter your OpenAI API key (will not be stored): ')
client = OpenAI(api_key=api_key)

print(f'REPO_ROOT       : {REPO_ROOT}')
print(f'DATASET_DIR     : {DATASET_DIR}  (exists: {DATASET_DIR.exists()})')
print(f'PROMPTS_DIR     : {PROMPTS_DIR}  (exists: {PROMPTS_DIR.exists()})')
print(f'STRATIFIED_LIST : {STRATIFIED_LIST}  (exists: {STRATIFIED_LIST.exists()})')
print(f'OUTPUT_CSV      : {OUTPUT_CSV}')
print(f'MODEL_ID        : {MODEL_ID}')
print(f'SAMPLE_SIZE     : {"100 (full stratified)" if SAMPLE_SIZE_OVERRIDE is None else SAMPLE_SIZE_OVERRIDE}')


## 2. Load the two prompt variants and the sample list

We load Variant A (Baseline) and Variant E (Full Framework, HINTED) — the same prompt files used in `02_run_ablation_hinted.ipynb` — and the canonical 100-sample stratified list. If `SAMPLE_SIZE_OVERRIDE` is set, we subsample deterministically within each category to preserve the stratification proportions.

In [ ]:
def load_prompt(filename: str) -> str:
    """Read a prompt file and strip a single trailing newline."""
    with open(PROMPTS_DIR / filename, encoding='utf-8') as f:
        return f.read().rstrip('\n')

PROMPTS = {
    'Variant_A_Baseline': load_prompt('variant_A_baseline.txt'),
    'Variant_E_Full':     load_prompt('variant_E_full_hinted.txt'),
}
SYSTEM_MESSAGE = load_prompt('system_message.txt')

for name, text in PROMPTS.items():
    print(f'{name:25s} ({len(text):3d} chars): {text[:80]}{"..." if len(text) > 80 else ""}')
print()
print(f'{"SYSTEM_MESSAGE":25s} ({len(SYSTEM_MESSAGE):3d} chars): {SYSTEM_MESSAGE[:80]}...')
print()

# Read the canonical 100-sample stratified subset (relative paths under final_dataset/).
with open(STRATIFIED_LIST) as f:
    subset_relpaths = [line.strip() for line in f if line.strip()]
print(f'Canonical stratified subset size: {len(subset_relpaths)}')

subset_paths = [DATASET_DIR / rp for rp in subset_relpaths]
missing = [p for p in subset_paths if not p.exists()]
if missing:
    raise RuntimeError(
        f'{len(missing)} samples listed in {STRATIFIED_LIST.name} are not present under {DATASET_DIR}. '
        f'Run 01_dataset_curation.ipynb first. Example missing: {missing[0]}'
    )

def get_true_label(filepath: Path) -> tuple:
    """Infer (true_label, true_cwe) from the parent directory name."""
    folder = filepath.parent.name
    if 'Safe' in folder:
        return 'Safe', None
    if '89' in folder:
        return 'Vulnerable', 'CWE-89'
    if '78' in folder:
        return 'Vulnerable', 'CWE-78'
    raise ValueError(f'Cannot infer label from folder name: {folder}')

# Optionally subsample to SAMPLE_SIZE_OVERRIDE while preserving category proportions.
if SAMPLE_SIZE_OVERRIDE is not None and SAMPLE_SIZE_OVERRIDE < len(subset_paths):
    import random
    from collections import defaultdict
    by_cat = defaultdict(list)
    for p in subset_paths:
        _, cwe = get_true_label(p)
        key = cwe if cwe else 'Safe'
        by_cat[key].append(p)
    target_total = SAMPLE_SIZE_OVERRIDE
    orig_total   = len(subset_paths)
    rng = random.Random(STRATIFY_SEED)
    selected = []
    for key, paths in by_cat.items():
        k = max(1, round(len(paths) * target_total / orig_total))
        rng.shuffle(paths)
        selected.extend(sorted(paths[:k]))
    subset_paths = sorted(selected)
    print(f'Subsampled to {len(subset_paths)} (seed={STRATIFY_SEED}, category proportions preserved).')

# Report the final category breakdown.
from collections import Counter
cat_counts = Counter()
for p in subset_paths:
    _, cwe = get_true_label(p)
    cat_counts[cwe if cwe else 'Safe'] += 1
print(f'Final sample: {len(subset_paths)} files -> {dict(cat_counts)}')


## 3. Run the frontier-model validation

For each sample, query GPT-5.5 twice (Variant A, then Variant E) and record the prediction. The output CSV is written incrementally after every sample, so the run can be safely interrupted and resumed.

**Reasoning-model robustness.** GPT-5.5 is a reasoning model and may (a) reject the `temperature` parameter, or (b) wrap its JSON in extra prose. The `predict()` function below handles both: it retries without `temperature` if the first call raises a parameter error, and it extracts the first JSON object from the response if direct parsing fails.

In [ ]:
import re

def _extract_prediction(content: str) -> str:
    """Parse a 'prediction' field from the model's response, tolerant of extra prose."""
    # First try: direct JSON parse.
    try:
        return json.loads(content).get('prediction', 'Error')
    except Exception:
        pass
    # Fallback: find the first {...} block and parse it.
    m = re.search(r'\{.*?\}', content, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(0)).get('prediction', 'Error')
        except Exception:
            pass
    # Last resort: keyword scan.
    low = content.lower()
    if 'vulnerable' in low and 'safe' not in low:
        return 'Vulnerable'
    if 'safe' in low and 'vulnerable' not in low:
        return 'Safe'
    return 'Error'

def predict(prompt_text: str, code_content: str) -> str:
    """Send one (prompt, code) pair to GPT-5.5 with retry. Return 'Vulnerable' / 'Safe' / 'Error'."""
    messages = [
        {'role': 'system', 'content': SYSTEM_MESSAGE},
        {'role': 'user',   'content': f'{prompt_text}\n\nTarget Code:\n{code_content}'},
    ]
    use_temperature = True
    for attempt in range(MAX_RETRIES):
        try:
            kwargs = {
                'model': MODEL_ID,
                'response_format': {'type': 'json_object'},
                'messages': messages,
            }
            if use_temperature:
                kwargs['temperature'] = TEMPERATURE
            response = client.chat.completions.create(**kwargs)
            return _extract_prediction(response.choices[0].message.content)
        except Exception as e:
            msg = str(e).lower()
            # If the model rejects 'temperature', drop it and retry immediately.
            if 'temperature' in msg and use_temperature:
                use_temperature = False
                continue
            if attempt + 1 < MAX_RETRIES:
                time.sleep(RETRY_BACKOFF_SEC)
            else:
                return 'Error'

# Resume from prior run if applicable.
if OUTPUT_CSV.exists():
    results_df = pd.read_csv(OUTPUT_CSV)
    processed = set(results_df['File_Name'].tolist())
    print(f'Resuming from existing CSV: {len(processed)} samples already processed.')
else:
    columns = ['File_Name', 'True_Label', 'True_CWE'] + list(PROMPTS.keys())
    results_df = pd.DataFrame(columns=columns)
    processed = set()

to_process = [p for p in subset_paths if p.name not in processed]
print(f'Samples remaining to process: {len(to_process)}')

for filepath in tqdm(to_process, desc=f'Frontier validation on {MODEL_ID}'):
    true_label, true_cwe = get_true_label(filepath)
    code_content = filepath.read_text(encoding='utf-8', errors='ignore')

    row = {'File_Name': filepath.name, 'True_Label': true_label, 'True_CWE': true_cwe}
    for variant_name, prompt_text in PROMPTS.items():
        row[variant_name] = predict(prompt_text, code_content)

    results_df = pd.concat([results_df, pd.DataFrame([row])], ignore_index=True)
    results_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

print(f'\nFrontier validation complete. Results written to {OUTPUT_CSV}')


## 4. Inspect results

Quick sanity checks: row count, error count, and per-variant prediction distribution. The key comparison — whether Variant E (Full Framework) under-performs Variant A (Baseline) on GPT-5.5, mirroring the Instruction Overload effect seen on GPT-4o-mini — is computed in detail (Accuracy, F1, Recall, Specificity, and the paired McNemar test) in `05_metrics_and_figures.ipynb`, which reads `rq4_frontier_gpt55.csv` alongside the other result files. Do **not** interpret the headline finding from this cell alone; use the metrics notebook.

In [ ]:
df = pd.read_csv(OUTPUT_CSV)

print(f'Total rows           : {len(df)}')
print()
print('True label distribution:')
print(df['True_Label'].value_counts().to_string())
print()
print('True CWE distribution:')
print(df['True_CWE'].value_counts(dropna=False).to_string())
print()
print('Per-variant prediction distribution:')
for variant in PROMPTS.keys():
    counts = df[variant].value_counts().to_dict()
    print(f'  {variant:25s} {counts}')
print()

# Error check: any cells that failed to parse.
n_errors = int((df[list(PROMPTS.keys())] == 'Error').sum().sum())
if n_errors:
    print(f'WARNING: {n_errors} Error cell(s) present. Re-run the run cell to retry them '
          f'(delete those rows from the CSV first, or delete the CSV to start over).')
else:
    print('No Error cells — all predictions parsed cleanly.')

# Quick directional peek (NOT the final statistic — see 05_metrics_and_figures.ipynb).
print()
print('Quick directional peek (informal; the paired McNemar test in notebook 05 is authoritative):')
for variant in PROMPTS.keys():
    correct = ((df[variant] == 'Vulnerable') & (df['True_Label'] == 'Vulnerable')) | \
              ((df[variant] == 'Safe') & (df['True_Label'] == 'Safe'))
    acc = correct.mean()
    print(f'  {variant:25s} crude accuracy = {acc:.3f}')
